# YOLO11n-Seg SimAM + CA Strong Augmentation — TigerShrimp Mixed Split

This standalone module trains the SimAM+CA architecture on the team-approved
TigerShrimp/ShrimpDisBD mixed split under two policies: the light augmentation
of the clean baseline and the stronger augmentation of the SimAM+CA source
notebook. Convention-matched images are grouped by `disease::shrimp_id`;
unmatched images are stratified by polygon-label stratum, then all partitions
are merged into train/valid/test.


In [ ]:
# Verify T4 GPU — Runtime > Change runtime type > GPU > T4
import subprocess
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(out.stdout if out.returncode == 0 else
      'No GPU detected. Enable T4: Runtime > Change runtime type > T4 GPU')


## 1. Download the new YOLO segmentation dataset


In [ ]:
# Team-approved Roboflow download: TigerShrimp / ShrimpDisBD v1 segmentation data.
from pathlib import Path

WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else (Path('/content') if Path('/content').exists() else Path.cwd())
ROBOFLOW_WORKSPACE = 'lets-try-this'
ROBOFLOW_PROJECT = 'shrimpdisbd-tigershrimp_mrtudat'
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = 'yolo26'

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec('roboflow') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])

from roboflow import Roboflow

ROBOFLOW_API_KEY_DIRECT = ''  # Keep empty. Use a Kaggle Secret named ROBOFLOW_API_KEY.

def get_roboflow_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
        if key:
            return key.strip()
    except Exception:
        pass
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    return os.environ.get('ROBOFLOW_API_KEY', '').strip()

api_key = get_roboflow_api_key()
if not api_key:
    raise RuntimeError('Missing ROBOFLOW_API_KEY. Add it to Kaggle Secrets; do not paste it into the notebook.')

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT)
DATASET_LOCATION = Path(dataset.location)
RAW_YOLO_DATASET_PATH = DATASET_LOCATION
print('Downloaded dataset location:', DATASET_LOCATION)


## 2. Team-approved mixed split

The next cell uses the agreed policy exactly. It intentionally does not replace
the unmatched-image branch with near-duplicate grouping.


In [ ]:
# Team-approved mixed split: grouped `disease::shrimp_id` for convention-matched images;
# stratified random by actual mask-label stratum for unmatched images; then merge.

## Mixed split: grouped for convention-matched names, stratified random for unmatched names

import hashlib
import os
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import yaml

SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

DATASET_DIR = Path(DATASET_LOCATION)
if not (DATASET_DIR / 'data.yaml').exists():
    yaml_candidates = sorted(DATASET_DIR.rglob('data.yaml'))
    if not yaml_candidates:
        raise FileNotFoundError(f'No data.yaml found below {DATASET_DIR}')
    DATASET_DIR = yaml_candidates[0].parent
base_path = DATASET_DIR
data_yaml_path = DATASET_DIR / 'data.yaml'

SHRIMP_NAME_PATTERN = re.compile(
    r'^(?P<disease>Healthy|BG|WSSV_BG|WSSV)-(?P<shrimp_id>.+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)

yaml_content = yaml.safe_load(data_yaml_path.read_text(encoding='utf-8')) or {}
raw_names = yaml_content.get('names', {})
if isinstance(raw_names, list):
    CLASS_NAMES = {i: str(name) for i, name in enumerate(raw_names)}
else:
    CLASS_NAMES = {int(k): str(v) for k, v in raw_names.items()}

def normalize_roboflow_stem(stem):
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)

def parse_shrimp_group_key(image_name):
    stem = normalize_roboflow_stem(Path(image_name).stem)
    match = SHRIMP_NAME_PATTERN.match(stem)
    if not match:
        return None, 'unparsed', None, None
    disease = match.group('disease')
    shrimp_id = match.group('shrimp_id')
    return f'{disease.lower()}::{shrimp_id}', disease, shrimp_id, int(match.group('img_num'))

def image_files_in_split(split):
    directory = base_path / split / 'images'
    if not directory.exists():
        return []
    return sorted(p for p in directory.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)

def label_path_for_image(image_path):
    path = image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'
    if path.exists():
        return path
    matches = sorted((image_path.parent.parent / 'labels').glob(f'{image_path.stem}*.txt'))
    return matches[0] if matches else path

def move_image_and_label(image_path, target_split):
    image_dst = base_path / target_split / 'images' / image_path.name
    label_src = label_path_for_image(image_path)
    label_dst = base_path / target_split / 'labels' / f'{image_path.stem}.txt'
    image_dst.parent.mkdir(parents=True, exist_ok=True)
    label_dst.parent.mkdir(parents=True, exist_ok=True)
    if image_path.resolve() != image_dst.resolve():
        if image_dst.exists():
            raise FileExistsError(f'Duplicate image destination: {image_dst}')
        shutil.move(str(image_path), str(image_dst))
    if label_src.exists() and label_src.resolve() != label_dst.resolve():
        if label_dst.exists():
            raise FileExistsError(f'Duplicate label destination: {label_dst}')
        shutil.move(str(label_src), str(label_dst))
    elif not label_dst.exists():
        label_dst.write_text('', encoding='utf-8')

def rebuild_train_pool_from_all_splits():
    all_images = []
    for split in ['train', 'valid', 'test']:
        all_images.extend(image_files_in_split(split))
    for image_path in all_images:
        move_image_and_label(image_path, 'train')
    return image_files_in_split('train')

def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob('**/*.cache'):
        cache_path.unlink()

def image_class_ids(image_path):
    label_path = label_path_for_image(image_path)
    ids = []
    if label_path.exists():
        for line in label_path.read_text(encoding='utf-8').splitlines():
            parts = line.strip().split()
            if parts:
                ids.append(int(float(parts[0])))
    return sorted(set(ids))

def disease_stratum_for_image(image_path):
    ids = image_class_ids(image_path)
    if not ids:
        return 'healthy_empty'
    names = [CLASS_NAMES.get(i, f'class_{i}') for i in ids]
    return '+'.join(name.strip().lower().replace(' ', '_') for name in names)

def split_one_stratum(items, seed, rng=None):
    items = list(items)
    (rng or random.Random(seed)).shuffle(items)
    n = len(items)
    n_train = int(TRAIN_RATIO * n)
    n_valid = int(VAL_RATIO * n)
    if n >= 3:
        n_valid = max(1, n_valid)
        n_train = max(1, n_train)
        if n_train + n_valid >= n:
            n_train = max(1, n - n_valid - 1)
    return {
        'train': items[:n_train],
        'valid': items[n_train:n_train + n_valid],
        'test': items[n_train + n_valid:],
    }

def split_grouped_items(group_items, seed):
    by_stratum = defaultdict(list)
    for group_key, filenames, stratum in group_items:
        by_stratum[stratum].append((group_key, filenames, stratum))
    out = {'train': [], 'valid': [], 'test': []}
    rng = random.Random(seed)
    for stratum, items in sorted(by_stratum.items()):
        pieces = split_one_stratum(sorted(items, key=lambda x: x[0]), seed + 11, rng=rng)
        for split, values in pieces.items():
            out[split].extend(values)
        print(f'Grouped branch {stratum}: {len(pieces["train"])} train groups, {len(pieces["valid"])} valid groups, {len(pieces["test"])} test groups')
    return out

def split_random_items(image_paths, seed):
    by_stratum = defaultdict(list)
    for path in image_paths:
        by_stratum[disease_stratum_for_image(path)].append(path.name)
    out = {'train': [], 'valid': [], 'test': []}
    for stratum, items in sorted(by_stratum.items()):
        pieces = split_one_stratum(sorted(items), seed + 29)
        for split, values in pieces.items():
            out[split].extend(values)
        print(f'Unmatched branch {stratum}: {len(pieces["train"])} train images, {len(pieces["valid"])} valid images, {len(pieces["test"])} test images')
    return out

def mixed_split(seed=42):
    image_paths = rebuild_train_pool_from_all_splits()
    convention_groups = defaultdict(list)
    unmatched = []
    source_meta = {}
    for path in image_paths:
        group_key, filename_disease, shrimp_id, img_num = parse_shrimp_group_key(path.name)
        if group_key is None:
            unmatched.append(path)
            source_meta[path.name] = {'source_subset': 'unmatched_stratified_random', 'naming_convention_matched': False, 'parsed_group_key': f'unparsed::{path.stem}', 'filename_disease': 'unparsed'}
        else:
            convention_groups[group_key].append(path.name)
            source_meta[path.name] = {'source_subset': 'convention_grouped_stratified', 'naming_convention_matched': True, 'parsed_group_key': group_key, 'filename_disease': filename_disease}

    grouped_items = []
    for group_key, filenames in convention_groups.items():
        # Preserve the original baseline's filename-disease stratification for recognized groups.
        strata = Counter(str(source_meta[name]['filename_disease']).lower() for name in filenames)
        grouped_items.append((group_key, sorted(filenames), strata.most_common(1)[0][0]))

    grouped_splits = split_grouped_items(grouped_items, seed)
    random_splits = split_random_items(unmatched, seed)
    assignments = {}
    for branch_splits in [grouped_splits, random_splits]:
        for split, items in branch_splits.items():
            values = items
            if values and isinstance(values[0], tuple):
                names = [name for _, filenames, _ in values for name in filenames]
            else:
                names = values
            for name in names:
                if name in assignments:
                    raise RuntimeError(f'Image assigned twice: {name}')
                assignments[name] = split

    if len(assignments) != len(image_paths):
        missing = sorted({p.name for p in image_paths} - set(assignments))
        raise RuntimeError(f'Mixed split did not assign every image. Missing: {missing[:10]}')

    for name, split in assignments.items():
        move_image_and_label(base_path / 'train' / 'images' / name, split)
    remove_yolo_label_caches(base_path)

    global SOURCE_META
    SOURCE_META = source_meta
    print(f'Mixed split complete: {len(convention_groups)} convention groups and {len(unmatched)} unmatched images.')
    print('Merged image counts:', {split: len(image_files_in_split(split)) for split in ['train', 'valid', 'test']})
    return assignments

mixed_split(SEED)

# The Roboflow export starts with every image in train. Rewrite paths only after
# the approved mixed split has finished moving images and labels.
data_yaml_path = Path(data_yaml_path)
data_yaml_content = yaml.safe_load(data_yaml_path.read_text(encoding='utf-8')) or {}
data_yaml_content['train'] = str((base_path / 'train' / 'images').resolve())
data_yaml_content['val'] = str((base_path / 'valid' / 'images').resolve())
data_yaml_content['test'] = str((base_path / 'test' / 'images').resolve())
data_yaml_path.write_text(
    yaml.safe_dump(data_yaml_content, sort_keys=False, allow_unicode=True),
    encoding='utf-8',
)
print('Updated prepared data.yaml:', data_yaml_path)



## Setup YOLO and prepared dataset paths


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

from ultralytics import YOLO

if "base_path" not in globals() or "data_yaml_path" not in globals():
    raise RuntimeError("Run the team-approved mixed split cell before model setup.")
base_path = str(Path(base_path))
data_yaml_path = str(Path(data_yaml_path))
if not Path(data_yaml_path).is_file():
    raise FileNotFoundError(f"Prepared data.yaml not found: {data_yaml_path}. Run the split cell first.")

YOLO_MODELS = [
    "yolo11n-seg.pt",
]
YOLO_MODEL = YOLO_MODELS[0]
MODEL_STEM = Path(YOLO_MODEL).stem
RUN_BASE_NAME = f"{MODEL_STEM}_shrimp_seg_clean_baseline"
model = YOLO(YOLO_MODEL)

print("Configured segmentation model:", YOLO_MODEL)
print("base_path:", base_path)
print("data_yaml_path:", data_yaml_path)


## 3. Prepared dataset configuration

The mixed-split cell above rewrites `data.yaml` to the newly created train, valid and test folders.


## 4. EDA


In [ ]:
import os
import yaml
import matplotlib.pyplot as plt
import cv2
import numpy as np
from collections import Counter
from pathlib import Path

# Load class names from data.yaml. Empty label files are healthy shrimp negatives.
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', [])
HEALTHY_CLASS_NAME = 'healthy'
print(f"Disease mask classes found: {class_names}")
print(f"Empty label files will be treated as: {HEALTHY_CLASS_NAME} shrimp negatives")


def split_label_stats(label_dir):
    instance_counts = Counter()
    labeled_images = 0
    healthy_images = 0
    missing_or_empty = 0
    label_dir = Path(label_dir)
    for label_file in label_dir.glob('*.txt'):
        lines = [line.strip() for line in label_file.read_text().splitlines() if line.strip()]
        if not lines:
            healthy_images += 1
            continue
        labeled_images += 1
        for line in lines:
            class_id = int(float(line.split()[0]))
            instance_counts[class_id] += 1
    return {
        'instance_counts': instance_counts,
        'labeled_images': labeled_images,
        'healthy_images': healthy_images,
        'total_label_files': labeled_images + healthy_images,
    }


stats = {}
for split in ['train', 'valid', 'test']:
    label_dir = os.path.join(base_path, split, 'labels')
    stats[split] = split_label_stats(label_dir)

for split, split_stats in stats.items():
    print(f"\n{split.capitalize()} Split:")
    print(f"  - labeled disease images: {split_stats['labeled_images']}")
    print(f"  - healthy negative images: {split_stats['healthy_images']}")
    for cid, count in split_stats['instance_counts'].items():
        name = class_names[cid] if cid < len(class_names) else f"Unknown({cid})"
        print(f"  - {name}: {count} mask instances")


## 5. Class imbalance visualization


In [ ]:
import pandas as pd
import seaborn as sns

instance_plot_data = []
image_plot_data = []
for split, split_stats in stats.items():
    image_plot_data.append({'Split': split, 'Class': HEALTHY_CLASS_NAME, 'Images': split_stats['healthy_images']})
    image_plot_data.append({'Split': split, 'Class': 'diseased_labeled', 'Images': split_stats['labeled_images']})
    for cid, count in split_stats['instance_counts'].items():
        instance_plot_data.append({'Split': split, 'Class': class_names[cid], 'Instances': count})

if instance_plot_data:
    df_instances = pd.DataFrame(instance_plot_data)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_instances, x='Split', y='Instances', hue='Class')
    plt.title('Disease Mask Instance Distribution across Splits')
    plt.show()

if image_plot_data:
    df_images = pd.DataFrame(image_plot_data)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_images, x='Split', y='Images', hue='Class')
    plt.title('Healthy Negative vs Diseased-Labeled Image Counts')
    plt.show()

for split, split_stats in stats.items():
    total_instances = sum(split_stats['instance_counts'].values())
    blackgill_ratio = (split_stats['instance_counts'].get(0, 0) / max(1, total_instances)) * 100
    healthy_ratio = (split_stats['healthy_images'] / max(1, split_stats['total_label_files'])) * 100
    print(f"{split.capitalize()}: {blackgill_ratio:.2f}% blackgill instances; {healthy_ratio:.2f}% healthy negative images")


## 6. Patch attention modules


In [ ]:
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())
import subprocess
import sys
import py_compile

ULTRALYTICS_REF = "v8.4.61"
ULTRA_DIR = WORK_ROOT / "ultralytics-v8.4.61"

if not ULTRA_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", ULTRALYTICS_REF, "https://github.com/ultralytics/ultralytics.git", str(ULTRA_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ULTRA_DIR)], check=True)

conv_py = ULTRA_DIR / "ultralytics/nn/modules/conv.py"
init_py = ULTRA_DIR / "ultralytics/nn/modules/__init__.py"
tasks_py = ULTRA_DIR / "ultralytics/nn/tasks.py"

for p in [conv_py, init_py, tasks_py]:
    backup = p.with_suffix(p.suffix + ".bak_all_attention_group_run")
    if not backup.exists():
        backup.write_text(p.read_text())
        print("Backup created:", backup)

attention_code = """
# =========================================================
# Custom attention modules for YOLO11 segmentation experiments
# =========================================================

class SimAM(nn.Module):
    def __init__(self, c1=None, e_lambda=1e-4):
        super().__init__()
        self.e_lambda = e_lambda
        self.activation = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w - 1
        if n <= 0:
            return x
        x_minus_mu_square = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        y = x_minus_mu_square / (
            4 * (x_minus_mu_square.sum(dim=[2, 3], keepdim=True) / n + self.e_lambda)
        ) + 0.5
        return x * self.activation(y)


class CoordAtt(nn.Module):
    def __init__(self, c1, c2=None, reduction=32):
        super().__init__()
        c2 = c1 if c2 is None else c2
        mip = max(8, c1 // reduction)
        self.conv1 = nn.Conv2d(c1, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()
        self.conv_h = nn.Conv2d(mip, c2, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, c2, kernel_size=1, stride=1, padding=0)
        self.proj = None
        if c1 != c2:
            self.proj = nn.Conv2d(c1, c2, kernel_size=1, stride=1, padding=0)

    def forward(self, x):
        identity = x
        n, c, h, w = x.size()
        x_h = x.mean(dim=3, keepdim=True)
        x_w = x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        a_h = self.conv_h(x_h).sigmoid()
        a_w = self.conv_w(x_w).sigmoid()
        if self.proj is not None:
            identity = self.proj(identity)
        return identity * a_h * a_w


class ECAAttention(nn.Module):
    def __init__(self, c1, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        y = self.avg_pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = self.sigmoid(y).transpose(-1, -2).unsqueeze(-1)
        return x * y.expand_as(x)


class CBAMAttention(nn.Module):
    def __init__(self, c1, c2=None, reduction=16, kernel_size=7):
        super().__init__()
        c2 = c1 if c2 is None else c2
        hidden = max(c1 // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(c1, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, c1, kernel_size=1, bias=False),
        )
        self.channel_sigmoid = nn.Sigmoid()
        assert kernel_size in (3, 7), "CBAM spatial kernel_size should be 3 or 7"
        padding = 3 if kernel_size == 7 else 1
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.spatial_sigmoid = nn.Sigmoid()
        self.proj = None
        if c1 != c2:
            self.proj = nn.Conv2d(c1, c2, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        ca = self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x))
        x = x * self.channel_sigmoid(ca)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        sa = torch.cat([avg_out, max_out], dim=1)
        x = x * self.spatial_sigmoid(self.spatial_conv(sa))
        if self.proj is not None:
            x = self.proj(x)
        return x


class EMAAttention(nn.Module):
    def __init__(self, c1, c2=None, groups=8):
        super().__init__()
        c2 = c1 if c2 is None else c2
        groups = max(1, int(groups))
        groups = min(groups, c1)
        if c1 % groups != 0:
            valid_groups = [g for g in [8, 4, 2, 1] if c1 % g == 0]
            groups = valid_groups[0] if valid_groups else 1
        self.c1 = c1
        self.c2 = c2
        self.groups = groups
        self.group_channels = c1 // groups
        self.softmax = nn.Softmax(dim=-1)
        self.agp = nn.AdaptiveAvgPool2d((1, 1))
        self.conv1x1 = nn.Conv2d(self.group_channels, self.group_channels, kernel_size=1, stride=1, padding=0)
        self.conv3x3 = nn.Conv2d(self.group_channels, self.group_channels, kernel_size=3, stride=1, padding=1)
        self.gn = nn.GroupNorm(self.group_channels, self.group_channels)
        self.proj = None
        if c1 != c2:
            self.proj = nn.Conv2d(c1, c2, kernel_size=1, stride=1, padding=0, bias=False)

    def forward(self, x):
        b, c, h, w = x.size()
        if c != self.c1:
            return x
        if c % self.groups != 0:
            return x
        group_x = x.reshape(b * self.groups, self.group_channels, h, w)
        x_h = group_x.mean(dim=3, keepdim=True)
        x_w = group_x.mean(dim=2, keepdim=True).permute(0, 1, 3, 2)
        hw = self.conv1x1(torch.cat([x_h, x_w], dim=2))
        x_h, x_w = torch.split(hw, [h, w], dim=2)
        x1 = self.gn(group_x * x_h.sigmoid() * x_w.permute(0, 1, 3, 2).sigmoid())
        x2 = self.conv3x3(group_x)
        x11 = self.softmax(self.agp(x1).reshape(b * self.groups, self.group_channels, 1).permute(0, 2, 1))
        x12 = x2.reshape(b * self.groups, self.group_channels, h * w)
        x21 = self.softmax(self.agp(x2).reshape(b * self.groups, self.group_channels, 1).permute(0, 2, 1))
        x22 = x1.reshape(b * self.groups, self.group_channels, h * w)
        weights = (torch.matmul(x11, x12) + torch.matmul(x21, x22)).reshape(b * self.groups, 1, h, w)
        out = (group_x * weights.sigmoid()).reshape(b, c, h, w)
        if self.proj is not None:
            out = self.proj(out)
        return out
"""

# =========================================================
# Patch conv.py
# =========================================================
text = conv_py.read_text()
text = text.replace(r'\"\"\"', '"""')

if "import torch\n" not in text:
    text = text.replace("import math\n", "import math\nimport torch\n") if "import math\n" in text else "import torch\n" + text

if "import torch.nn as nn\n" not in text and "from torch import nn\n" not in text:
    text = text.replace("import torch\n", "import torch\nimport torch.nn as nn\n")

for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
    token = f'"{module_name}",'
    if token not in text:
        if '"RepConv",\n    "SpatialAttention",' in text:
            text = text.replace(
                '    "RepConv",\n    "SpatialAttention",',
                f'    "RepConv",\n    "{module_name}",\n    "SpatialAttention",'
            )
        elif '"Concat",' in text:
            text = text.replace('"Concat",', f'"Concat",\n    "{module_name}",')

missing_any = any(
    f"class {m}(nn.Module):" not in text
    for m in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]
)
if missing_any:
    text = text.rstrip() + "\n" + attention_code + "\n"

conv_py.write_text(text)
print("Patched conv.py with attention modules")

try:
    py_compile.compile(str(conv_py), doraise=True)
    print("conv.py syntax OK")
except Exception as e:
    print("conv.py syntax error:")
    print(e)
    lines = conv_py.read_text().splitlines()
    line_no = getattr(getattr(e, "exc_value", None), "lineno", 1)
    start = max(0, line_no - 8)
    end = min(len(lines), line_no + 8)
    print(f"\n--- conv.py lines {start + 1} to {end} ---")
    for i in range(start, end):
        print(f"{i + 1}: {lines[i]}")
    raise

# =========================================================
# Patch __init__.py
# =========================================================
text = init_py.read_text()

if "from .conv import (" in text:
    for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
        conv_import_block = text.split("from .conv import (", 1)[1].split(")", 1)[0]
        if module_name not in conv_import_block:
            if "    RepConv,\n    SpatialAttention," in text:
                text = text.replace(
                    "    RepConv,\n    SpatialAttention,",
                    f"    RepConv,\n    {module_name},\n    SpatialAttention,"
                )
            elif "    Concat," in text:
                text = text.replace("    Concat,", f"    Concat,\n    {module_name},")
            else:
                text = text.replace("from .conv import (", f"from .conv import (\n    {module_name},")
else:
    for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
        if f"from .conv import {module_name}" not in text:
            text += f"\nfrom .conv import {module_name}\n"

if "__all__" in text:
    for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
        all_block = text.split("__all__", 1)[1]
        if f'"{module_name}"' not in all_block:
            if '"SemanticSegment",\n    "SpatialAttention",' in text:
                text = text.replace(
                    '    "SemanticSegment",\n    "SpatialAttention",',
                    f'    "SemanticSegment",\n    "{module_name}",\n    "SpatialAttention",'
                )
            elif '"Concat",' in text:
                text = text.replace('"Concat",', f'"Concat",\n    "{module_name}",')

init_py.write_text(text)
print("Patched __init__.py")

# =========================================================
# Patch tasks.py
# =========================================================
text = tasks_py.read_text()

if "from ultralytics.nn.modules import (" in text:
    for module_name in ["SimAM", "CoordAtt", "ECAAttention", "CBAMAttention", "EMAAttention"]:
        modules_import_block = text.split("from ultralytics.nn.modules import (", 1)[1].split(")", 1)[0]
        if module_name not in modules_import_block:
            if "    Segment,\n    Segment26," in text:
                text = text.replace(
                    "    Segment,\n    Segment26,",
                    f"    Segment,\n    Segment26,\n    {module_name},"
                )
            elif "    Concat," in text:
                text = text.replace("    Concat,", f"    Concat,\n    {module_name},")
            else:
                text = text.replace(
                    "from ultralytics.nn.modules import (",
                    f"from ultralytics.nn.modules import (\n    {module_name},"
                )

base_start = text.find("base_modules = frozenset")
repeat_start = text.find("repeat_modules = frozenset", base_start)
if base_start == -1 or repeat_start == -1:
    raise RuntimeError("Could not locate base_modules block in tasks.py")

def add_to_base_modules(text, module_name):
    base_start = text.find("base_modules = frozenset")
    repeat_start = text.find("repeat_modules = frozenset", base_start)
    base_block = text[base_start:repeat_start]
    if module_name in base_block:
        print(f"{module_name} already exists in base_modules")
        return text
    pattern1 = "base_modules = frozenset(\n        {"
    pattern2 = "base_modules = frozenset({"
    if pattern1 in text:
        text = text.replace(pattern1, f"base_modules = frozenset(\n        {{\n            {module_name},")
        print(f"Added {module_name} to base_modules using pattern1")
    elif pattern2 in text:
        text = text.replace(pattern2, f"base_modules = frozenset({{\n            {module_name},")
        print(f"Added {module_name} to base_modules using pattern2")
    else:
        raise RuntimeError(f"Could not insert {module_name} into base_modules.")
    return text

for module_name in ["CoordAtt", "CBAMAttention", "EMAAttention"]:
    text = add_to_base_modules(text, module_name)

# ECA and SimAM keep channel count; parse manually.
if "elif m is ECAAttention:" not in text:
    simam_anchor = """elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]"""
    eca_branch = """elif m is ECAAttention:
            c2 = ch[f]
            if len(args) >= 2:
                k_size = args[1]
            elif len(args) == 1:
                k_size = args[0]
            else:
                k_size = 3
            args = [c2, k_size]
        elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]"""
    if simam_anchor in text:
        text = text.replace(simam_anchor, eca_branch)
    else:
        detect_anchor = """elif m in frozenset(
            {
                Detect,"""
        eca_and_simam_branch = """elif m is ECAAttention:
            c2 = ch[f]
            if len(args) >= 2:
                k_size = args[1]
            elif len(args) == 1:
                k_size = args[0]
            else:
                k_size = 3
            args = [c2, k_size]
        elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]
        """
        if detect_anchor in text:
            text = text.replace(detect_anchor, eca_and_simam_branch + detect_anchor)
        else:
            raise RuntimeError("Could not find insertion point for ECAAttention/SimAM parse branch.")

if "elif m is SimAM:" not in text:
    detect_anchor = """elif m in frozenset(
            {
                Detect,"""
    simam_branch = """elif m is SimAM:
            c2 = ch[f]
            args = [c2, *args]
        """
    if detect_anchor in text:
        text = text.replace(detect_anchor, simam_branch + detect_anchor)
    else:
        raise RuntimeError("Could not find insertion point for SimAM parse branch.")

tasks_py.write_text(text)
print("Patched tasks.py")

print("Attention registration complete.")
print("IMPORTANT: Restart runtime/kernel before creating YOLO models.")


In [ ]:
# =========================================================
# Force using local patched Ultralytics repo
# =========================================================
import sys
import subprocess
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())

ULTRALYTICS_REF = "v8.4.61"
ULTRA_DIR = WORK_ROOT / "ultralytics-v8.4.61"

if not ULTRA_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", ULTRALYTICS_REF, "https://github.com/ultralytics/ultralytics.git", str(ULTRA_DIR)],
        check=True
    )

# Install local editable repo
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(ULTRA_DIR)],
    check=True
)

# Put local repo at the front of Python import path
sys.path.insert(0, str(ULTRA_DIR))

# Clear previously imported pip ultralytics modules
for module_name in list(sys.modules.keys()):
    if module_name == "ultralytics" or module_name.startswith("ultralytics."):
        del sys.modules[module_name]

import ultralytics
from ultralytics import YOLO

print("Using Ultralytics from:")
print(ultralytics.__file__)

assert str(ULTRA_DIR) in ultralytics.__file__, (
    "Still importing wrong Ultralytics version. "
    f"Expected local repo: {ULTRA_DIR}, got: {ultralytics.__file__}"
)


## 7. Create all attention YAML files


In [ ]:
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())

ULTRA_DIR = WORK_ROOT / "ultralytics-v8.4.61"
MODEL_CFG_DIR = ULTRA_DIR / "ultralytics/cfg/models/11"
MODEL_CFG_DIR.mkdir(parents=True, exist_ok=True)

BASE_YAML_TEMPLATE = """nc: 2
scales:
  n: [0.50, 0.25, 1024]
  s: [0.50, 0.50, 1024]
  m: [0.50, 1.00, 512]
  l: [1.00, 1.00, 512]
  x: [1.00, 1.50, 512]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]
  - [-1, 2, C2PSA, [1024]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

{attention_layers}
"""

YAML_LAYER_DEFS = {
    "simam": ("yolo11n-seg-simam-head.yaml", """  # SimAM before Segment head.
  - [16, 1, SimAM, []]
  - [19, 1, SimAM, []]
  - [22, 1, SimAM, []]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "ca": ("yolo11n-seg-ca-head.yaml", """  # Coordinate Attention before Segment head.
  - [16, 1, CoordAtt, [256, 32]]
  - [19, 1, CoordAtt, [512, 32]]
  - [22, 1, CoordAtt, [1024, 32]]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "eca": ("yolo11n-seg-eca-head.yaml", """  # ECA before Segment head.
  - [16, 1, ECAAttention, [3]]
  - [19, 1, ECAAttention, [3]]
  - [22, 1, ECAAttention, [3]]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "cbam": ("yolo11n-seg-cbam-head.yaml", """  # CBAM before Segment head.
  - [16, 1, CBAMAttention, [256, 16, 7]]
  - [19, 1, CBAMAttention, [512, 16, 7]]
  - [22, 1, CBAMAttention, [1024, 16, 7]]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "ema": ("yolo11n-seg-ema-head.yaml", """  # EMA before Segment head.
  - [16, 1, EMAAttention, [256, 8]]
  - [19, 1, EMAAttention, [512, 8]]
  - [22, 1, EMAAttention, [1024, 8]]
  - [[23, 24, 25], 1, Segment, [nc, 32, 256]]"""),
    "simam_ca": ("yolo11n-seg-simam-ca-head.yaml", """  # CA -> SimAM before Segment head.
  - [16, 1, CoordAtt, [256, 32]]
  - [23, 1, SimAM, []]

  - [19, 1, CoordAtt, [512, 32]]
  - [25, 1, SimAM, []]

  - [22, 1, CoordAtt, [1024, 32]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""),
    "eca_simam": ("yolo11n-seg-eca-simam-head.yaml", """  # ECA -> SimAM before Segment head.
  - [16, 1, ECAAttention, [3]]
  - [23, 1, SimAM, []]

  - [19, 1, ECAAttention, [3]]
  - [25, 1, SimAM, []]

  - [22, 1, ECAAttention, [3]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""),
    "ca_ema": ("yolo11n-seg-ca-ema-head.yaml", """  # CA -> EMA before Segment head.
  - [16, 1, CoordAtt, [256, 32]]
  - [23, 1, EMAAttention, [256, 8]]

  - [19, 1, CoordAtt, [512, 32]]
  - [25, 1, EMAAttention, [512, 8]]

  - [22, 1, CoordAtt, [1024, 32]]
  - [27, 1, EMAAttention, [1024, 8]]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""),
    "cbam_simam": ("yolo11n-seg-cbam-simam-head.yaml", """  # CBAM -> SimAM before Segment head.
  - [16, 1, CBAMAttention, [256, 16, 7]]
  - [23, 1, SimAM, []]

  - [19, 1, CBAMAttention, [512, 16, 7]]
  - [25, 1, SimAM, []]

  - [22, 1, CBAMAttention, [1024, 16, 7]]
  - [27, 1, SimAM, []]

  - [[24, 26, 28], 1, Segment, [nc, 32, 256]]"""),
    "ema_simam_ca": ("yolo11n-seg-ema-simam-ca-head.yaml", """  # CA -> EMA -> SimAM before Segment head.
  - [16, 1, CoordAtt, [256, 32]]
  - [23, 1, EMAAttention, [256, 8]]
  - [24, 1, SimAM, []]

  - [19, 1, CoordAtt, [512, 32]]
  - [26, 1, EMAAttention, [512, 8]]
  - [27, 1, SimAM, []]

  - [22, 1, CoordAtt, [1024, 32]]
  - [29, 1, EMAAttention, [1024, 8]]
  - [30, 1, SimAM, []]

  - [[25, 28, 31], 1, Segment, [nc, 32, 256]]"""),
}

MODEL_YAML_PATHS = {}
for key, (yaml_name, layers) in YAML_LAYER_DEFS.items():
    yaml_text = BASE_YAML_TEMPLATE.format(attention_layers=layers)
    yaml_path = MODEL_CFG_DIR / yaml_name
    yaml_path.write_text(yaml_text)
    MODEL_YAML_PATHS[key] = str(yaml_path)
    print("Created:", key, "->", yaml_path)


## 8. Select experiment group


In [ ]:
import sys
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())

sys.path.insert(0, str(WORK_ROOT / "ultralytics-v8.4.61"))

EXPERIMENT_ROOT_STR = str(WORK_ROOT / "yolo11n_simam_ca_mixed_split_run")
RUN_BASE_NAME = "simam_ca_mixed_split_run"

EXPERIMENTS = [
    {
        "key": "simam_ca_light",
        "name": "YOLO11n-seg + SimAM + CA | baseline light augmentation",
        "augmentation_policy": "light",
        "model_type": "attention",
        "yaml": MODEL_YAML_PATHS["simam_ca"],
    },
    {
        "key": "simam_ca_strong",
        "name": "YOLO11n-seg + SimAM + CA | SimAM-CA strong augmentation",
        "augmentation_policy": "strong",
        "model_type": "attention",
        "yaml": MODEL_YAML_PATHS["simam_ca"],
    },
]

print("Selected experiments:")
for experiment in EXPERIMENTS:
    print("-", experiment["key"], "|", experiment["augmentation_policy"])


## 9. Train all models and build summary


### Required preflight: paths, segmentation labels, and model shapes

This gate runs immediately before any training. It stops the notebook if a split path is invalid, an image has no matching label file, a polygon label is malformed, or the configured model cannot complete a dummy segmentation forward pass.


In [ ]:
# Mandatory preflight gate. Do not bypass this cell before a real training run.
from pathlib import Path
import math

import torch
import yaml
from ultralytics import YOLO

PREFLIGHT_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def _preflight_image_dir(value, data_config_path):
    """Resolve a YOLO data.yaml split value to its images directory."""
    if isinstance(value, (list, tuple)):
        if len(value) != 1:
            raise ValueError("This notebook requires exactly one image directory per split.")
        value = value[0]
    image_dir = Path(str(value))
    if not image_dir.is_absolute():
        image_dir = data_config_path.parent / image_dir
    return image_dir.resolve()


def _preflight_validate_dataset(data_config_path):
    data_config_path = Path(data_config_path).resolve()
    if not data_config_path.is_file():
        raise FileNotFoundError(f"Prepared data.yaml was not found: {data_config_path}")

    config = yaml.safe_load(data_config_path.read_text(encoding="utf-8")) or {}
    names = config.get("names", {})
    class_count = len(names) if isinstance(names, (list, tuple, dict)) else 0
    if class_count < 1:
        raise ValueError("data.yaml must define at least one class in `names`.")

    summary = {}
    for split_name in ("train", "val", "test"):
        if split_name not in config:
            raise KeyError(f"data.yaml is missing the `{split_name}` split.")
        image_dir = _preflight_image_dir(config[split_name], data_config_path)
        label_dir = image_dir.parent / "labels"
        if not image_dir.is_dir() or not label_dir.is_dir():
            raise FileNotFoundError(
                f"{split_name}: expected images={image_dir} and labels={label_dir}"
            )

        images = sorted(
            path for path in image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in PREFLIGHT_IMAGE_EXTENSIONS
        )
        if not images:
            raise RuntimeError(f"{split_name}: image split is empty: {image_dir}")

        label_files = sorted(path for path in label_dir.glob("*.txt") if path.is_file())
        image_stems = {path.stem for path in images}
        label_stems = {path.stem for path in label_files}
        missing_labels = sorted(image_stems - label_stems)
        orphan_labels = sorted(label_stems - image_stems)
        if missing_labels or orphan_labels:
            raise RuntimeError(
                f"{split_name}: image/label mismatch; "
                f"missing_labels={missing_labels[:5]}, orphan_labels={orphan_labels[:5]}"
            )

        labeled_images = 0
        polygons = 0
        for label_path in label_files:
            lines = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            if lines:
                labeled_images += 1
            for line_number, line in enumerate(lines, start=1):
                tokens = line.split()
                if len(tokens) < 7 or (len(tokens) - 1) % 2 != 0:
                    raise ValueError(
                        f"{split_name}: malformed segmentation polygon at "
                        f"{label_path.name}:{line_number}"
                    )
                try:
                    class_id = int(tokens[0])
                    coordinates = [float(token) for token in tokens[1:]]
                except ValueError as exc:
                    raise ValueError(
                        f"{split_name}: non-numeric label at {label_path.name}:{line_number}"
                    ) from exc
                if not 0 <= class_id < class_count:
                    raise ValueError(
                        f"{split_name}: class id {class_id} out of range at "
                        f"{label_path.name}:{line_number}"
                    )
                if any(not math.isfinite(value) or value < 0.0 or value > 1.0 for value in coordinates):
                    raise ValueError(
                        f"{split_name}: polygon coordinate outside [0, 1] at "
                        f"{label_path.name}:{line_number}"
                    )
                polygons += 1
        summary[split_name] = {
            "images": len(images),
            "empty_labels": len(images) - labeled_images,
            "polygons": polygons,
        }
    return summary


def _preflight_flatten_tensors(value):
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _preflight_flatten_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _preflight_flatten_tensors(item)


def run_notebook_preflight(model_specs, expected_attention=None, dummy_imgsz=128):
    """Fail before training on a bad dataset path, label, model build, or tensor shape."""
    if dummy_imgsz % 32:
        raise ValueError("dummy_imgsz must be divisible by 32 for YOLO segmentation.")
    dataset_summary = _preflight_validate_dataset(data_yaml_path)
    print("Dataset preflight PASS:", dataset_summary)

    checked_specs = []
    for model_spec in dict.fromkeys(str(spec) for spec in model_specs):
        model = YOLO(model_spec)
        module = model.model.eval()
        attention_counts = {}
        attention_shape_errors = []
        attention_handles = []
        shape_preserving_names = {
            "CoTEBoundaryLiteGate", "DPCAGate", "LargeKernelAttention", "CoordAtt", "SimAM"
        }

        def _attention_shape_hook(layer_name):
            def hook(_module, inputs, output):
                input_tensor = inputs[0] if inputs else None
                if not torch.is_tensor(input_tensor) or not torch.is_tensor(output):
                    attention_shape_errors.append(f"{layer_name}: non-tensor attention input/output")
                elif input_tensor.ndim != 4 or output.ndim != 4 or tuple(input_tensor.shape) != tuple(output.shape):
                    attention_shape_errors.append(
                        f"{layer_name}: expected shape-preserving BCHW, got "
                        f"{tuple(input_tensor.shape)} -> {tuple(output.shape)}"
                    )
            return hook

        for layer in module.modules():
            layer_name = layer.__class__.__name__
            if layer_name in shape_preserving_names:
                attention_counts[layer_name] = attention_counts.get(layer_name, 0) + 1
                attention_handles.append(layer.register_forward_hook(_attention_shape_hook(layer_name)))
        try:
            device = next(module.parameters()).device
        except StopIteration:
            device = torch.device("cpu")
        dummy = torch.zeros(1, 3, dummy_imgsz, dummy_imgsz, device=device)
        with torch.inference_mode():
            output = module(dummy)
        for handle in attention_handles:
            handle.remove()
        if attention_shape_errors:
            raise RuntimeError(f"{model_spec}: attention shape check failed: {attention_shape_errors}")
        if expected_attention is not None:
            observed_attention = {name: attention_counts.get(name, 0) for name in expected_attention}
            if observed_attention != expected_attention:
                raise RuntimeError(
                    f"{model_spec}: attention module count mismatch; "
                    f"expected={expected_attention}, observed={observed_attention}"
                )
        tensors = list(_preflight_flatten_tensors(output))
        if not tensors:
            raise RuntimeError(f"{model_spec}: model forward returned no tensors.")
        if any(tensor.numel() == 0 or not torch.isfinite(tensor).all().item() for tensor in tensors):
            raise FloatingPointError(f"{model_spec}: model forward produced empty or non-finite tensors.")
        checked_specs.append(model_spec)
        del model, module, dummy, output, tensors
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("Model build/shape preflight PASS:", checked_specs)
    return dataset_summary


In [ ]:
import csv
import gc
import math
import os
import random
import shutil
import time
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from ultralytics import YOLO

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())

# =========================================================
# Reproducibility
# =========================================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print("Deterministic warning:", e)

# =========================================================
# Paths and clean augmentation
# =========================================================
EXPERIMENT_ROOT = Path(EXPERIMENT_ROOT_STR)
RUNS_DIR = WORK_ROOT / "runs" / "segment"
REPORT_DIR = EXPERIMENT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

COUNT_PENALTY_WEIGHT = 0.05
DISEASE_MISS_PENALTY_WEIGHT = 0.15
HEALTHY_FP_PENALTY_WEIGHT = 0.10
PREDICT_CONF_FOR_COUNT = 0.25

LIGHT_TRAIN_ARGS = {
    # Exact light policy used by the clean YOLO baseline.
    "auto_augment": None,
    "erasing": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "fliplr": 0.5,
    "flipud": 0.0,
    "hsv_h": 0.01,
    "hsv_s": 0.35,
    "hsv_v": 0.20,
    "degrees": 0.0,
    "translate": 0.05,
    "scale": 0.20,
    "shear": 0.0,
    "perspective": 0.0,
    "multi_scale": 0.0,
    "bgr": 0.0,
}

STRONG_TRAIN_ARGS = {
    # Exact strong policy from yolov11n_simam_augmentation.ipynb.
    "auto_augment": None,
    "erasing": 0.15,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "fliplr": 0.5,
    "flipud": 0.3,
    "hsv_h": 0.05,
    "hsv_s": 0.50,
    "hsv_v": 0.40,
    "degrees": 10.0,
    "translate": 0.10,
    "scale": 0.50,
    "shear": 0.0,
    "perspective": 0.0,
    "multi_scale": 0.0,
    "bgr": 0.0,
}

AUGMENTATION_POLICIES = {
    "light": LIGHT_TRAIN_ARGS,
    "strong": STRONG_TRAIN_ARGS,
}


IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
# Dataset created in Cell 5; never fall back to the old shrimpDisHandSegV2 data.
if "base_path" not in globals() or "data_yaml_path" not in globals():
    raise RuntimeError("Run the team-approved mixed split cell before training.")
base_path = str(Path(base_path))
data_yaml_path = str(Path(data_yaml_path))

# =========================================================
# Disable default Ultralytics Albumentations hook if available
# =========================================================
try:
    import ultralytics.data.augment as yolo_aug
    if hasattr(yolo_aug, "Albumentations"):
        class NoAlbumentations:
            def __init__(self, *args, **kwargs):
                pass
            def __call__(self, labels):
                return labels
        yolo_aug.Albumentations = NoAlbumentations
        print("Disabled default Ultralytics Albumentations hook.")
except Exception as e:
    print("Could not disable Albumentations hook:", e)

# =========================================================
# Utility functions
# =========================================================
def remove_yolo_label_caches(root):
    for cache_path in Path(root).glob("**/*.cache"):
        cache_path.unlink()
        print(f"Removed stale cache: {cache_path}")

def count_labeled_images(label_dir):
    labeled = 0
    healthy = 0
    instances = 0
    for label_path in Path(label_dir).glob("*.txt"):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if lines:
            labeled += 1
            instances += len(lines)
        else:
            healthy += 1
    return {"labeled_images": labeled, "healthy_images": healthy, "instances": instances}

def write_data_yaml(dataset_dir, yaml_path, val_dir="valid", test_dir="test"):
    with open(data_yaml_path, "r") as f:
        content = yaml.safe_load(f)
    content["train"] = str(Path(dataset_dir) / "train" / "images")
    content["val"] = str(Path(dataset_dir) / val_dir / "images")
    content["test"] = str(Path(dataset_dir) / test_dir / "images")
    with open(yaml_path, "w") as f:
        yaml.safe_dump(content, f, sort_keys=False)
    return yaml_path

def copy_dataset_for_experiment(exp_key):
    src = Path(base_path)
    dst = EXPERIMENT_ROOT / exp_key / "dataset"
    if dst.exists():
        shutil.rmtree(dst)
    ignore = shutil.ignore_patterns("runs", "*.cache", ".clahe_applied")
    shutil.copytree(src, dst, ignore=ignore)
    remove_yolo_label_caches(dst)
    return dst

def find_image_for_label(image_dir, label_name):
    stem = Path(label_name).stem
    for ext in IMAGE_EXTENSIONS:
        candidate = Path(image_dir) / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None

def copy_split_by_label_state(src_dataset, dst_dataset, split, want_labeled):
    src_images = Path(src_dataset) / split / "images"
    src_labels = Path(src_dataset) / split / "labels"
    dst_images = Path(dst_dataset) / split / "images"
    dst_labels = Path(dst_dataset) / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)
    copied = 0
    for label_path in sorted(src_labels.glob("*.txt")):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        is_labeled = bool(lines)
        if is_labeled != want_labeled:
            continue
        image_path = find_image_for_label(src_images, label_path.name)
        if image_path is None:
            continue
        shutil.copy2(image_path, dst_images / image_path.name)
        shutil.copy2(label_path, dst_labels / label_path.name)
        copied += 1
    return copied

def make_state_eval_dataset(src_dataset, exp_key, state_name, want_labeled):
    dst = EXPERIMENT_ROOT / exp_key / f"dataset_{state_name}_eval"
    if dst.exists():
        shutil.rmtree(dst)
    for sub in ["images", "labels"]:
        (dst / "train" / sub).mkdir(parents=True, exist_ok=True)
    copied = {}
    for split in ["valid", "test"]:
        copied[split] = copy_split_by_label_state(src_dataset, dst, split, want_labeled=want_labeled)
    yaml_path = dst / f"data_{state_name}.yaml"
    write_data_yaml(dst, yaml_path)
    print(f"{state_name} eval dataset for {exp_key}: {copied}")
    return dst, yaml_path, copied

def make_labeled_only_eval_dataset(src_dataset, exp_key):
    return make_state_eval_dataset(src_dataset, exp_key, "labeled_only", want_labeled=True)

def make_healthy_only_eval_dataset(src_dataset, exp_key):
    return make_state_eval_dataset(src_dataset, exp_key, "healthy_only", want_labeled=False)

def metric_value(metrics, dotted_path, default=float("nan")):
    obj = metrics
    for part in dotted_path.split("."):
        if not hasattr(obj, part):
            return default
        obj = getattr(obj, part)
    try:
        return float(obj)
    except Exception:
        return default

def list_images(image_dir):
    image_dir = Path(image_dir)
    images = []
    for ext in IMAGE_EXTENSIONS:
        images.extend(image_dir.rglob(f"*{ext}"))
    return sorted(images)

def image_to_label_path(image_path):
    return Path(image_path).parent.parent / "labels" / f"{Path(image_path).stem}.txt"

def read_label_instances(label_path):
    label_path = Path(label_path)
    if not label_path.exists():
        return []
    rows = []
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        try:
            cls = int(float(parts[0]))
            nums = [float(x) for x in parts[1:]]
        except Exception:
            continue
        rows.append((cls, nums))
    return rows

def instance_to_box(instance, image_path):
    cls, nums = instance
    w, h = Image.open(image_path).size
    if len(nums) == 4:
        xc, yc, bw, bh = nums
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h
    else:
        coords = np.array(nums, dtype=float).reshape(-1, 2)
        xs = coords[:, 0] * w
        ys = coords[:, 1] * h
        x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
    return [float(x1), float(y1), float(x2), float(y2), int(cls)]

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a[:4]
    bx1, by1, bx2, by2 = b[:4]
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def get_class_names(yaml_path):
    with open(yaml_path, "r") as f:
        cfg = yaml.safe_load(f)
    names = cfg.get("names", {})
    if isinstance(names, list):
        return {i: n for i, n in enumerate(names)}
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    return {}

def count_prediction_errors(model, images_dir, labels_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    if not image_paths:
        return {
            "images": 0, "gt_total": 0, "pred_box_total": 0, "pred_mask_total": 0,
            "box_count_mae": float("nan"), "mask_count_mae": float("nan"),
            "box_count_exact": float("nan"), "mask_count_exact": float("nan"),
            "disease_images": 0, "disease_box_miss_images": 0, "disease_mask_miss_images": 0,
            "disease_box_miss_rate": float("nan"), "disease_mask_miss_rate": float("nan")
        }
    results = model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, verbose=False)
    box_errors, mask_errors, box_exact, mask_exact = [], [], [], []
    gt_total = pred_box_total = pred_mask_total = 0
    disease_images = disease_box_miss_images = disease_mask_miss_images = 0
    for image_path, result in zip(image_paths, results):
        label_path = Path(labels_dir) / f"{image_path.stem}.txt"
        gt_count = len([line for line in label_path.read_text().splitlines() if line.strip()]) if label_path.exists() else 0
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        denom = max(1, gt_count)
        box_errors.append(abs(box_count - gt_count) / denom)
        mask_errors.append(abs(mask_count - gt_count) / denom)
        box_exact.append(float(box_count == gt_count))
        mask_exact.append(float(mask_count == gt_count))
        if gt_count > 0:
            disease_images += 1
            disease_box_miss_images += int(box_count == 0)
            disease_mask_miss_images += int(mask_count == 0)
        gt_total += gt_count
        pred_box_total += box_count
        pred_mask_total += mask_count
    return {
        "images": len(image_paths), "gt_total": gt_total,
        "pred_box_total": pred_box_total, "pred_mask_total": pred_mask_total,
        "box_count_mae": sum(box_errors) / len(box_errors),
        "mask_count_mae": sum(mask_errors) / len(mask_errors),
        "box_count_exact": sum(box_exact) / len(box_exact),
        "mask_count_exact": sum(mask_exact) / len(mask_exact),
        "disease_images": disease_images,
        "disease_box_miss_images": disease_box_miss_images,
        "disease_mask_miss_images": disease_mask_miss_images,
        "disease_box_miss_rate": disease_box_miss_images / disease_images if disease_images else float("nan"),
        "disease_mask_miss_rate": disease_mask_miss_images / disease_images if disease_images else float("nan"),
    }

def healthy_false_positive_summary(model, images_dir, conf=PREDICT_CONF_FOR_COUNT):
    image_paths = list_images(images_dir)
    if not image_paths:
        return {
            "healthy_images": 0, "healthy_images_with_box_fp": 0, "healthy_images_with_mask_fp": 0,
            "healthy_box_fp_rate": float("nan"), "healthy_mask_fp_rate": float("nan"),
            "healthy_fp_boxes_total": 0, "healthy_fp_masks_total": 0,
            "healthy_fp_boxes_per_image": float("nan"), "healthy_fp_masks_per_image": float("nan"),
            "healthy_avg_fp_confidence": float("nan"),
        }
    results = model.predict(source=[str(p) for p in image_paths], imgsz=640, conf=conf, verbose=False)
    images_with_box_fp = images_with_mask_fp = box_total = mask_total = 0
    confidences = []
    for result in results:
        box_count = len(result.boxes) if result.boxes is not None else 0
        mask_count = len(result.masks) if result.masks is not None else 0
        if box_count > 0:
            images_with_box_fp += 1
            try:
                confidences.extend([float(v) for v in result.boxes.conf.detach().cpu().tolist()])
            except Exception:
                pass
        if mask_count > 0:
            images_with_mask_fp += 1
        box_total += box_count
        mask_total += mask_count
    n = len(image_paths)
    return {
        "healthy_images": n,
        "healthy_images_with_box_fp": images_with_box_fp,
        "healthy_images_with_mask_fp": images_with_mask_fp,
        "healthy_box_fp_rate": images_with_box_fp / n,
        "healthy_mask_fp_rate": images_with_mask_fp / n,
        "healthy_fp_boxes_total": box_total,
        "healthy_fp_masks_total": mask_total,
        "healthy_fp_boxes_per_image": box_total / n,
        "healthy_fp_masks_per_image": mask_total / n,
        "healthy_avg_fp_confidence": sum(confidences) / len(confidences) if confidences else 0.0,
    }

def read_best_epoch_from_results(run_path):
    results_csv = Path(run_path) / "results.csv"
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    mask_col = "metrics/mAP50(M)"
    if mask_col not in df.columns:
        return {"epochs_ran": len(df)}
    best_idx = df[mask_col].idxmax()
    first = df.iloc[0]
    best = df.iloc[best_idx]
    last = df.iloc[-1]
    return {
        "epochs_ran": int(len(df)),
        "best_epoch_by_mask_map50": int(best["epoch"]) if "epoch" in df.columns else int(best_idx + 1),
        "first_train_seg_loss": float(first.get("train/seg_loss", float("nan"))),
        "best_val_mask_map50": float(best.get(mask_col, float("nan"))),
        "best_val_mask_map50_95": float(best.get("metrics/mAP50-95(M)", float("nan"))),
        "last_val_mask_map50": float(last.get(mask_col, float("nan"))),
        "last_val_mask_map50_95": float(last.get("metrics/mAP50-95(M)", float("nan"))),
        "last_train_seg_loss": float(last.get("train/seg_loss", float("nan"))),
        "last_val_seg_loss": float(last.get("val/seg_loss", float("nan"))),
        "seg_loss_gap_val_minus_train": float(last.get("val/seg_loss", float("nan")) - last.get("train/seg_loss", float("nan"))),
    }

def class_level_test_analysis(model, dataset_dir, yaml_path, report_dir, conf=0.25, iou_threshold=0.50):
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    class_names = get_class_names(yaml_path)
    def cname(cid): return class_names.get(int(cid), str(cid))

    # Annotation count by class
    annotation_rows = []
    for split in ["train", "valid", "test"]:
        image_dir = Path(dataset_dir) / split / "images"
        instance_counter, image_counter = Counter(), Counter()
        for img in list_images(image_dir):
            instances = read_label_instances(image_to_label_path(img))
            seen = set()
            for inst in instances:
                cls = inst[0]
                instance_counter[cls] += 1
                seen.add(cls)
            for cls in seen:
                image_counter[cls] += 1
        for cls, count in sorted(instance_counter.items()):
            annotation_rows.append({
                "split": split,
                "class_id": cls,
                "class_name": cname(cls),
                "annotation_count": count,
                "image_count_with_class": image_counter[cls],
            })
    annotation_df = pd.DataFrame(annotation_rows)
    annotation_df.to_csv(report_dir / "annotation_count_by_class.csv", index=False)

    if not annotation_df.empty:
        pivot = annotation_df.pivot_table(index="class_name", columns="split", values="annotation_count", aggfunc="sum", fill_value=0)
        ax = pivot.plot(kind="bar", figsize=(10, 5))
        ax.set_title("Annotation count by class and split")
        ax.set_xlabel("Class")
        ax.set_ylabel("Annotation count")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(report_dir / "annotation_count_by_class.png", dpi=200)
        plt.close()

    # TEST confusion/miss/FP
    test_images = list_images(Path(dataset_dir) / "test" / "images")
    confusion_counter, miss_counter, fp_counter, matched_counter = Counter(), Counter(), Counter(), Counter()
    for img in test_images:
        gt_instances = read_label_instances(image_to_label_path(img))
        gt_boxes = [instance_to_box(inst, img) for inst in gt_instances]
        result = model.predict(str(img), imgsz=640, conf=conf, iou=0.5, verbose=False)[0]
        pred_boxes = []
        if result.boxes is not None and len(result.boxes) > 0:
            xyxy = result.boxes.xyxy.cpu().numpy()
            cls_arr = result.boxes.cls.cpu().numpy().astype(int)
            conf_arr = result.boxes.conf.cpu().numpy()
            for box, cls, score in zip(xyxy, cls_arr, conf_arr):
                pred_boxes.append([float(box[0]), float(box[1]), float(box[2]), float(box[3]), int(cls), float(score)])
        used_pred = set()
        for gt in gt_boxes:
            best_iou, best_j = 0.0, -1
            for j, pred in enumerate(pred_boxes):
                if j in used_pred:
                    continue
                iou = box_iou(gt, pred)
                if iou > best_iou:
                    best_iou, best_j = iou, j
            gt_cls = int(gt[4])
            if best_j >= 0 and best_iou >= iou_threshold:
                used_pred.add(best_j)
                pred_cls = int(pred_boxes[best_j][4])
                matched_counter[cname(gt_cls)] += 1
                if pred_cls != gt_cls:
                    confusion_counter[(cname(gt_cls), cname(pred_cls))] += 1
            else:
                miss_counter[cname(gt_cls)] += 1
        for j, pred in enumerate(pred_boxes):
            if j not in used_pred:
                fp_counter[cname(int(pred[4]))] += 1

    confusion_df = pd.DataFrame(
        [{"gt_class": gt, "pred_class": pred, "confused_count": count} for (gt, pred), count in confusion_counter.items()]
    ).sort_values("confused_count", ascending=False) if confusion_counter else pd.DataFrame(columns=["gt_class", "pred_class", "confused_count"])

    miss_df = pd.DataFrame(
        [{"class_name": cls, "missed_box_count": count} for cls, count in miss_counter.items()]
    ).sort_values("missed_box_count", ascending=False) if miss_counter else pd.DataFrame(columns=["class_name", "missed_box_count"])

    fp_df = pd.DataFrame(
        [{"class_name": cls, "false_positive_count": count} for cls, count in fp_counter.items()]
    ).sort_values("false_positive_count", ascending=False) if fp_counter else pd.DataFrame(columns=["class_name", "false_positive_count"])

    confusion_df.to_csv(report_dir / "test_class_confusion_counts.csv", index=False)
    miss_df.to_csv(report_dir / "test_missed_box_counts_by_class.csv", index=False)
    fp_df.to_csv(report_dir / "test_false_positive_counts_by_class.csv", index=False)

    if not confusion_df.empty:
        labels = confusion_df.apply(lambda r: f"{r['gt_class']} → {r['pred_class']}", axis=1)
        plt.figure(figsize=(10, max(4, 0.5 * len(labels))))
        plt.barh(labels, confusion_df["confused_count"])
        plt.title("TEST class confusion counts")
        plt.xlabel("Count")
        plt.ylabel("GT → Pred")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(report_dir / "test_class_confusion_counts.png", dpi=200)
        plt.close()

    if not miss_df.empty:
        plt.figure(figsize=(8, 4))
        plt.bar(miss_df["class_name"], miss_df["missed_box_count"])
        plt.title("TEST missed boxes by class")
        plt.xlabel("Class")
        plt.ylabel("Missed box count")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(report_dir / "test_missed_boxes_by_class.png", dpi=200)
        plt.close()

    return {
        "test_confused_total": int(confusion_df["confused_count"].sum()) if not confusion_df.empty else 0,
        "test_missed_box_total": int(miss_df["missed_box_count"].sum()) if not miss_df.empty else 0,
        "test_false_positive_total": int(fp_df["false_positive_count"].sum()) if not fp_df.empty else 0,
    }

def save_loss_and_correlation_plots(run_path, report_dir):
    report_dir = Path(report_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    results_csv = Path(run_path) / "results.csv"
    if not results_csv.exists():
        return {}
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    df.to_csv(report_dir / "results_clean_columns.csv", index=False)

    loss_cols = [c for c in df.columns if "loss" in c.lower()]
    if loss_cols:
        plt.figure(figsize=(12, 6))
        x = df["epoch"] if "epoch" in df.columns else df.index
        for col in loss_cols:
            plt.plot(x, df[col], label=col)
        plt.title("Train / validation loss curves")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(report_dir / "train_val_loss_curves.png", dpi=200)
        plt.close()

    numeric_df = df.select_dtypes(include=[np.number])
    if numeric_df.shape[1] >= 2:
        corr = numeric_df.corr()
        corr.to_csv(report_dir / "training_metrics_correlation_matrix.csv")
        plt.figure(figsize=(12, 10))
        im = plt.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
        plt.colorbar(im, fraction=0.046, pad=0.04)
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.columns)), corr.columns)
        plt.title("Correlation matrix of training metrics")
        plt.tight_layout()
        plt.savefig(report_dir / "training_metrics_correlation_matrix.png", dpi=200)
        plt.close()
    return {"results_csv": str(results_csv)}

def make_model(exp):
    if exp["model_type"] == "baseline":
        return YOLO("yolo11n-seg.pt")
    model = YOLO(exp["yaml"])
    try:
        model.load("yolo11n-seg.pt")
        print(f"Loaded pretrained weights for {exp['key']}")
    except Exception as e:
        print(f"Pretrained load warning for {exp['key']}:", e)
    return model

def run_experiment(exp):
    print("\n" + "#" * 90)
    print(f"Starting experiment: {exp['name']}")
    print("#" * 90)

    dataset_dir = copy_dataset_for_experiment(exp["key"])
    remove_yolo_label_caches(dataset_dir)
    yaml_path = dataset_dir / "data.yaml"
    write_data_yaml(dataset_dir, yaml_path)
    labeled_eval_dir, labeled_eval_yaml, _ = make_labeled_only_eval_dataset(dataset_dir, exp["key"])
    healthy_eval_dir, healthy_eval_yaml, _ = make_healthy_only_eval_dataset(dataset_dir, exp["key"])

    split_counts = {}
    for split in ["train", "valid", "test"]:
        split_counts[split] = count_labeled_images(dataset_dir / split / "labels")
        print(f"{exp['key']} {split}: {split_counts[split]}")

    run_name = f"{RUN_BASE_NAME}_{exp['key']}"
    yolo = make_model(exp)

    start = time.time()
    yolo.train(
        data=str(yaml_path),
        task="segment",
        imgsz=640,
        epochs=100,
        batch=16,
        patience=30,
        seed=42,
        deterministic=True,
        workers=0,
        project=str(RUNS_DIR),
        name=run_name,
        exist_ok=True,
        pretrained=True,
        plots=True,
        verbose=True,
        **AUGMENTATION_POLICIES[exp["augmentation_policy"]],
    )
    train_time_min = (time.time() - start) / 60

    run_path = RUNS_DIR / run_name
    best_path = run_path / "weights" / "best.pt"
    best_model = YOLO(str(best_path))

    # YOLO val-like reports
    full_val = best_model.val(data=str(yaml_path), split="val", imgsz=640, plots=True, verbose=False)
    full_test = best_model.val(data=str(yaml_path), split="test", imgsz=640, plots=True, verbose=False)
    labeled_val = best_model.val(data=str(labeled_eval_yaml), split="val", imgsz=640, plots=False, verbose=False)
    labeled_test = best_model.val(data=str(labeled_eval_yaml), split="test", imgsz=640, plots=False, verbose=False)

    labeled_val_count = count_prediction_errors(best_model, labeled_eval_dir / "valid" / "images", labeled_eval_dir / "valid" / "labels")
    healthy_val_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / "valid" / "images")
    test_count = count_prediction_errors(best_model, dataset_dir / "test" / "images", dataset_dir / "test" / "labels")
    labeled_test_count = count_prediction_errors(best_model, labeled_eval_dir / "test" / "images", labeled_eval_dir / "test" / "labels")
    healthy_test_fp = healthy_false_positive_summary(best_model, healthy_eval_dir / "test" / "images")

    extra_dir = Path(run_path) / "extra_test_metrics"
    class_metrics = class_level_test_analysis(best_model, dataset_dir, yaml_path, extra_dir)
    save_loss_and_correlation_plots(run_path, extra_dir)

    labeled_val_mask_map50 = metric_value(labeled_val, "seg.map50")
    val_count_penalty = COUNT_PENALTY_WEIGHT * labeled_val_count["mask_count_mae"]
    val_disease_miss_penalty = DISEASE_MISS_PENALTY_WEIGHT * labeled_val_count["disease_box_miss_rate"]
    val_healthy_fp_penalty = HEALTHY_FP_PENALTY_WEIGHT * healthy_val_fp["healthy_mask_fp_rate"]
    healthy_aware_val_score = labeled_val_mask_map50 - val_count_penalty - val_disease_miss_penalty - val_healthy_fp_penalty

    labeled_mask_map50 = metric_value(labeled_test, "seg.map50")
    count_penalty = COUNT_PENALTY_WEIGHT * labeled_test_count["mask_count_mae"]
    disease_miss_penalty = DISEASE_MISS_PENALTY_WEIGHT * labeled_test_count["disease_box_miss_rate"]
    healthy_fp_penalty = HEALTHY_FP_PENALTY_WEIGHT * healthy_test_fp["healthy_mask_fp_rate"]
    healthy_aware_score = labeled_mask_map50 - count_penalty - disease_miss_penalty - healthy_fp_penalty

    row = {
        "experiment": exp["key"],
        "name": exp["name"],
        "model_type": exp["model_type"],
        "augmentation_policy": exp["augmentation_policy"],
        "model": exp.get("yaml", "yolo11n-seg.pt"),
        "run_name": run_name,
        "run_path": str(run_path),
        "best_pt": str(best_path),
        "train_time_min": round(train_time_min, 2),

        # Full val/test YOLO report style
        "full_val_box_precision": metric_value(full_val, "box.mp"),
        "full_val_box_recall": metric_value(full_val, "box.mr"),
        "full_val_box_map50": metric_value(full_val, "box.map50"),
        "full_val_box_map50_95": metric_value(full_val, "box.map"),
        "full_val_mask_precision": metric_value(full_val, "seg.mp"),
        "full_val_mask_recall": metric_value(full_val, "seg.mr"),
        "full_val_mask_map50": metric_value(full_val, "seg.map50"),
        "full_val_mask_map50_95": metric_value(full_val, "seg.map"),

        "full_test_box_precision": metric_value(full_test, "box.mp"),
        "full_test_box_recall": metric_value(full_test, "box.mr"),
        "full_test_box_map50": metric_value(full_test, "box.map50"),
        "full_test_box_map50_95": metric_value(full_test, "box.map"),
        "full_test_mask_precision": metric_value(full_test, "seg.mp"),
        "full_test_mask_recall": metric_value(full_test, "seg.mr"),
        "full_test_mask_map50": metric_value(full_test, "seg.map50"),
        "full_test_mask_map50_95": metric_value(full_test, "seg.map"),

        # Labeled-only diseased test
        "labeled_val_box_map50": metric_value(labeled_val, "box.map50"),
        "labeled_val_mask_map50": labeled_val_mask_map50,
        "labeled_val_mask_count_mae": labeled_val_count["mask_count_mae"],
        "labeled_val_disease_box_miss_rate": labeled_val_count["disease_box_miss_rate"],
        "healthy_val_images": healthy_val_fp["healthy_images"],
        "healthy_val_mask_fp_rate": healthy_val_fp["healthy_mask_fp_rate"],
        "labeled_test_box_map50": metric_value(labeled_test, "box.map50"),
        "labeled_test_mask_map50": labeled_mask_map50,
        "labeled_test_mask_map50_95": metric_value(labeled_test, "seg.map"),

        # Count/miss/FP
        "test_gt_instances": test_count["gt_total"],
        "test_pred_boxes": test_count["pred_box_total"],
        "test_pred_masks": test_count["pred_mask_total"],
        "test_mask_count_mae": test_count["mask_count_mae"],
        "labeled_test_gt_instances": labeled_test_count["gt_total"],
        "labeled_test_pred_boxes": labeled_test_count["pred_box_total"],
        "labeled_test_pred_masks": labeled_test_count["pred_mask_total"],
        "labeled_test_mask_count_mae": labeled_test_count["mask_count_mae"],
        "labeled_test_mask_count_exact": labeled_test_count["mask_count_exact"],
        "labeled_test_disease_box_miss_rate": labeled_test_count["disease_box_miss_rate"],
        "labeled_test_disease_mask_miss_rate": labeled_test_count["disease_mask_miss_rate"],
        "labeled_test_disease_box_miss_images": labeled_test_count["disease_box_miss_images"],
        "labeled_test_disease_mask_miss_images": labeled_test_count["disease_mask_miss_images"],
        "healthy_test_images": healthy_test_fp["healthy_images"],
        "healthy_test_mask_fp_rate": healthy_test_fp["healthy_mask_fp_rate"],
        "healthy_test_box_fp_rate": healthy_test_fp["healthy_box_fp_rate"],
        "healthy_test_fp_masks_total": healthy_test_fp["healthy_fp_masks_total"],
        "healthy_test_fp_masks_per_image": healthy_test_fp["healthy_fp_masks_per_image"],
        "healthy_test_avg_fp_confidence": healthy_test_fp["healthy_avg_fp_confidence"],

        # Extra requested metrics
        **class_metrics,

        # Healthy-aware score
        "count_penalty": count_penalty,
        "disease_miss_penalty": disease_miss_penalty,
        "healthy_fp_penalty": healthy_fp_penalty,
        "healthy_aware_labeled_val_mask_map50": healthy_aware_val_score,
        "healthy_aware_labeled_test_mask_map50": healthy_aware_score,
    }
    row.update(read_best_epoch_from_results(run_path))

    del yolo, best_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

# =========================================================
# Run all experiments in this notebook
# =========================================================

# Hard gate: do not start any training until all paths, polygons and model shapes pass.
run_notebook_preflight([experiment["yaml"] for experiment in EXPERIMENTS], expected_attention={'CoordAtt': 3, 'SimAM': 3})

experiment_results = []
for experiment in EXPERIMENTS:
    result = run_experiment(experiment)
    experiment_results.append(result)
    partial_df = pd.DataFrame(experiment_results)
    display(partial_df)
    partial_df.to_csv(REPORT_DIR / "partial_summary.csv", index=False)

summary_df = pd.DataFrame(experiment_results)
summary_df = summary_df.sort_values("healthy_aware_labeled_val_mask_map50", ascending=False).reset_index(drop=True)
summary_csv = REPORT_DIR / "summary_all_models.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"Saved summary: {summary_csv}")
display(summary_df)

BEST_RUN = summary_df.iloc[0].to_dict()
print("Best run by validation healthy-aware score:", BEST_RUN["run_name"])
print("Best checkpoint:", BEST_RUN["best_pt"])


## 10. Visualize summary and locate reports


In [ ]:
# =========================================================
# Visualize TEST predictions for EACH module like baseline
# =========================================================
from ultralytics import YOLO
import glob
import matplotlib.pyplot as plt
import os
import random
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

# =========================================================
# Config
# =========================================================
SHOW_ONLY_BEST = False       # True = chỉ show model tốt nhất, False = show từng module
MAX_AUG_IMAGES = 6
MAX_LABELED_TEST_IMAGES = 8
MAX_HEALTHY_TEST_IMAGES = 8
PRED_CONF = 0.10
IMG_SIZE = 640
SEED = 42

random.seed(SEED)

# =========================================================
# Load summary
# =========================================================
summary_csv = REPORT_DIR / "summary_all_models.csv"

if not summary_csv.exists():
    raise FileNotFoundError(
        f"Không tìm thấy summary file: {summary_csv}\n"
        "Hãy chạy cell train/evaluate trước để tạo summary_all_models.csv."
    )

summary_df = pd.read_csv(summary_csv)
display(summary_df)

if SHOW_ONLY_BEST:
    summary_df = summary_df.head(1)

# =========================================================
# Helpers
# =========================================================
def draw_yolo_segmentation_labels(image_path, label_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(image_path)

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    overlay = image.copy()
    colors = [(255, 70, 70), (70, 180, 255), (90, 220, 120), (240, 180, 60)]

    if label_path.exists():
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 7:
                continue

            cls_id = int(float(parts[0]))
            coords = np.array([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
            coords[:, 0] *= width
            coords[:, 1] *= height
            pts = coords.astype(np.int32)

            color = colors[cls_id % len(colors)]
            cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=2)
            cv2.fillPoly(overlay, [pts], color=color)

            x, y = pts[0]

            try:
                label = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            except Exception:
                label = str(cls_id)

            cv2.putText(
                overlay,
                label,
                (int(x), int(y)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2,
            )

    return cv2.addWeighted(overlay, 0.35, image, 0.65, 0)


def has_nonempty_label(image_path, labels_dir):
    label_path = labels_dir / f"{image_path.stem}.txt"
    return label_path.exists() and bool(label_path.read_text().strip())


def sample_or_first(items, max_n, random_sample=False):
    items = list(items)
    if len(items) <= max_n:
        return items

    if random_sample:
        return random.sample(items, k=max_n)

    return items[:max_n]


# =========================================================
# Loop through each module/model
# =========================================================
for _, row in summary_df.iterrows():
    experiment = row["experiment"]
    run_name = row["run_name"]
    run_path = Path(row["run_path"])
    best_model_path = Path(row["best_pt"])

    selected_exp = experiment
    selected_dataset_dir = EXPERIMENT_ROOT / selected_exp / "dataset"
    selected_yaml = selected_dataset_dir / "data.yaml"

    selected_labeled_yaml = (
        EXPERIMENT_ROOT
        / selected_exp
        / "dataset_labeled_only_eval"
        / "data_labeled_only.yaml"
    )

    selected_healthy_dir = (
        EXPERIMENT_ROOT
        / selected_exp
        / "dataset_healthy_only_eval"
    )

    selected_healthy_yaml = selected_healthy_dir / "data_healthy_only.yaml"

    print("\n" + "=" * 120)
    print(f"Experiment: {experiment}")
    print(f"Run name: {run_name}")
    print(f"Best model: {best_model_path}")
    print(f"Dataset: {selected_dataset_dir}")
    print("=" * 120)

    if not best_model_path.exists():
        print(f"[SKIP] Missing best.pt: {best_model_path}")
        continue

    if not selected_dataset_dir.exists():
        print(f"[SKIP] Missing dataset dir: {selected_dataset_dir}")
        continue

    model_inference = YOLO(str(best_model_path))

    # =====================================================
    # 1. Full test-set evaluation
    # =====================================================
    print("\nFull test-set evaluation:")
    full_test_metrics = model_inference.val(
        data=str(selected_yaml),
        split="test",
        imgsz=IMG_SIZE,
        plots=True,
        verbose=False,
    )

    # =====================================================
    # 2. Labeled-only diseased test-set evaluation
    # =====================================================
    print("\nLabeled-only diseased test-set evaluation:")
    labeled_test_metrics = model_inference.val(
        data=str(selected_labeled_yaml),
        split="test",
        imgsz=IMG_SIZE,
        plots=False,
        verbose=False,
    )

    # =====================================================
    # 3. Healthy false-positive evaluation
    # =====================================================
    healthy_test_fp = healthy_false_positive_summary(
        model_inference,
        selected_healthy_dir / "test" / "images",
    )

    print("Full test mask mAP50:", metric_value(full_test_metrics, "seg.map50"))
    print("Labeled-only diseased test mask mAP50:", metric_value(labeled_test_metrics, "seg.map50"))
    print("Healthy test mask false-positive rate:", healthy_test_fp["healthy_mask_fp_rate"])
    print("Healthy test false-positive masks per image:", healthy_test_fp["healthy_fp_masks_per_image"])

    # =====================================================
    # 4. Visualize augmented training GT masks directly
    # =====================================================
    print("\nVisualize augmented training ground-truth masks directly.")

    aug_images = sorted((selected_dataset_dir / "train" / "images").glob("aug_*"))
    aug_images = sample_or_first(aug_images, MAX_AUG_IMAGES, random_sample=True)

    if aug_images:
        fig, axes = plt.subplots(
            len(aug_images),
            1,
            figsize=(10, 5 * len(aug_images)),
        )

        if len(aug_images) == 1:
            axes = [axes]

        for ax, image_path in zip(axes, aug_images):
            label_path = selected_dataset_dir / "train" / "labels" / f"{image_path.stem}.txt"

            ax.imshow(draw_yolo_segmentation_labels(image_path, label_path))
            ax.set_title(
                f"{experiment} | Augmented train GT mask: {image_path.name} | "
                f"label exists={label_path.exists()}"
            )
            ax.axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print("No augmented training images found to visualize.")

    # =====================================================
    # 5. Visualize labeled test predictions
    # =====================================================
    print("\nVisualize labeled test predictions.")

    test_images = []
    for ext in IMAGE_EXTENSIONS:
        test_images.extend((selected_dataset_dir / "test" / "images").glob(f"*{ext}"))

    test_labels_dir = selected_dataset_dir / "test" / "labels"

    labeled_test_images = [
        p for p in sorted(test_images)
        if has_nonempty_label(p, test_labels_dir)
    ]

    labeled_test_images = sample_or_first(
        labeled_test_images,
        MAX_LABELED_TEST_IMAGES,
        random_sample=False,
    )

    if labeled_test_images:
        results = model_inference.predict(
            source=[str(p) for p in labeled_test_images],
            conf=PRED_CONF,
            imgsz=IMG_SIZE,
            save=True,
            verbose=False,
        )

        for image_path, result in zip(labeled_test_images, results):
            plt.figure(figsize=(8, 8))
            plt.imshow(result.plot())
            plt.title(
                f"{experiment} | Labeled test prediction: {image_path.name} "
                f"(conf > {PRED_CONF})"
            )
            plt.axis("off")
            plt.show()
    else:
        print("No labeled test images found to visualize.")

    # =====================================================
    # 6. Visualize healthy negative predictions
    # =====================================================
    print("\nVisualize healthy negative predictions to inspect false positives.")

    healthy_images = []
    for ext in IMAGE_EXTENSIONS:
        healthy_images.extend((selected_healthy_dir / "test" / "images").glob(f"*{ext}"))

    healthy_images = sample_or_first(
        sorted(healthy_images),
        MAX_HEALTHY_TEST_IMAGES,
        random_sample=False,
    )

    if healthy_images:
        healthy_results = model_inference.predict(
            source=[str(p) for p in healthy_images],
            conf=PRED_CONF,
            imgsz=IMG_SIZE,
            save=False,
            verbose=False,
        )

        for image_path, result in zip(healthy_images, healthy_results):
            box_count = len(result.boxes) if result.boxes is not None else 0
            mask_count = len(result.masks) if result.masks is not None else 0

            plt.figure(figsize=(8, 8))
            plt.imshow(result.plot())
            plt.title(
                f"{experiment} | Healthy test prediction: {image_path.name} | "
                f"boxes={box_count}, masks={mask_count}"
            )
            plt.axis("off")
            plt.show()
    else:
        print("No healthy test images found to visualize.")

print("\nDone visualizing predictions for each module.")


In [ ]:
# =========================================================
# Package all experiment results into a ZIP file
# Works for both single_attention and combined_attention notebooks
# =========================================================
from pathlib import Path

if "WORK_ROOT" not in globals():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else (Path("/content") if Path("/content").exists() else Path.cwd())
import zipfile
import time
import shutil
import pandas as pd
import os

# =========================================================
# Config
# =========================================================
INCLUDE_WEIGHTS = True      # True = include best.pt/last.pt, False = skip weights to reduce zip size
INCLUDE_DATASET = False     # True = include copied experiment datasets, False = skip datasets
INCLUDE_RUNS = True         # Include YOLO run folders: results.csv, plots, confusion matrix, predictions...
INCLUDE_REPORTS = True      # Include reports folder: summary_all_models.csv, partial_summary.csv...
INCLUDE_EXTRA_METRICS = True

TIMESTAMP = time.strftime("%Y%m%d_%H%M%S")

# =========================================================
# Resolve paths
# =========================================================
if "REPORT_DIR" not in globals():
    raise NameError("REPORT_DIR is not defined. Please run the train/evaluate cell first.")

if "EXPERIMENT_ROOT" not in globals():
    raise NameError("EXPERIMENT_ROOT is not defined. Please run the train/evaluate cell first.")

REPORT_DIR = Path(REPORT_DIR)
EXPERIMENT_ROOT = Path(EXPERIMENT_ROOT)

summary_csv = REPORT_DIR / "summary_all_models.csv"

if not summary_csv.exists():
    raise FileNotFoundError(
        f"summary_all_models.csv not found at: {summary_csv}\n"
        "Please run the train/evaluate cell first."
    )

summary_df = pd.read_csv(summary_csv)

# Detect notebook type from EXPERIMENT_ROOT name
root_name = EXPERIMENT_ROOT.name
zip_name = f"{root_name}_all_results_{TIMESTAMP}.zip"
zip_path = WORK_ROOT / zip_name

print("Packaging results...")
print("EXPERIMENT_ROOT:", EXPERIMENT_ROOT)
print("REPORT_DIR:", REPORT_DIR)
print("summary_csv:", summary_csv)
print("zip_path:", zip_path)

# =========================================================
# Helper functions
# =========================================================
def should_skip_file(path: Path):
    path = Path(path)

    # Skip cache/temp files
    skip_suffixes = {
        ".cache",
        ".tmp",
        ".log",
    }

    if path.suffix.lower() in skip_suffixes:
        return True

    # Skip large weights if disabled
    if not INCLUDE_WEIGHTS and path.suffix.lower() == ".pt":
        return True

    # Skip dataset images/labels if disabled
    if not INCLUDE_DATASET:
        parts = set(path.parts)
        if "dataset" in parts:
            return True
        if "dataset_labeled_only_eval" in parts:
            return True
        if "dataset_healthy_only_eval" in parts:
            return True

    return False


def add_file_to_zip(zf, file_path: Path, arc_base: Path):
    file_path = Path(file_path)

    if not file_path.exists() or not file_path.is_file():
        return

    if should_skip_file(file_path):
        return

    try:
        arcname = file_path.relative_to(arc_base)
    except Exception:
        arcname = file_path.name

    zf.write(file_path, arcname=str(arcname))


def add_dir_to_zip(zf, dir_path: Path, arc_base: Path):
    dir_path = Path(dir_path)

    if not dir_path.exists():
        print(f"[SKIP] Missing dir: {dir_path}")
        return

    for file_path in dir_path.rglob("*"):
        if file_path.is_file():
            add_file_to_zip(zf, file_path, arc_base)


# =========================================================
# Create ZIP
# =========================================================
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:

    # -----------------------------------------------------
    # 1. Add reports folder
    # -----------------------------------------------------
    if INCLUDE_REPORTS and REPORT_DIR.exists():
        print("Adding REPORT_DIR...")
        add_dir_to_zip(zf, REPORT_DIR, REPORT_DIR.parent)

    # -----------------------------------------------------
    # 2. Add experiment root
    # Includes copied datasets only if INCLUDE_DATASET=True
    # -----------------------------------------------------
    if EXPERIMENT_ROOT.exists():
        print("Adding EXPERIMENT_ROOT...")
        add_dir_to_zip(zf, EXPERIMENT_ROOT, EXPERIMENT_ROOT.parent)

    # -----------------------------------------------------
    # 3. Add YOLO run folders from summary_all_models.csv
    # -----------------------------------------------------
    if INCLUDE_RUNS:
        print("Adding YOLO run folders from summary...")

        for _, row in summary_df.iterrows():
            experiment = row.get("experiment", "unknown")
            run_path = Path(row.get("run_path", ""))

            if not run_path.exists():
                print(f"[SKIP] Missing run_path for {experiment}: {run_path}")
                continue

            print(f"  - Adding run: {experiment} -> {run_path}")
            add_dir_to_zip(zf, run_path, run_path.parent)

    # -----------------------------------------------------
    # 4. Add selected important files explicitly
    # -----------------------------------------------------
    important_files = [
        summary_csv,
        REPORT_DIR / "partial_summary.csv",
        REPORT_DIR / "summary_full_test_mask_map50.png",
        REPORT_DIR / "summary_test_missed_box_total.png",
        REPORT_DIR / "summary_test_confused_total.png",
    ]

    for f in important_files:
        if f.exists():
            add_file_to_zip(zf, f, REPORT_DIR.parent)

    # -----------------------------------------------------
    # 5. Add README
    # -----------------------------------------------------
    readme_text = f"""
YOLO11n Attention Experiment Results Package
Generated at: {TIMESTAMP}

Experiment root:
{EXPERIMENT_ROOT}

Report directory:
{REPORT_DIR}

Summary file:
{summary_csv}

Included settings:
- INCLUDE_WEIGHTS = {INCLUDE_WEIGHTS}
- INCLUDE_DATASET = {INCLUDE_DATASET}
- INCLUDE_RUNS = {INCLUDE_RUNS}
- INCLUDE_REPORTS = {INCLUDE_REPORTS}
- INCLUDE_EXTRA_METRICS = {INCLUDE_EXTRA_METRICS}

Main expected files:
- reports/summary_all_models.csv
- reports/partial_summary.csv
- per-run results.csv
- per-run confusion_matrix.png
- per-run results.png
- per-run extra_test_metrics/
- per-run train_val_loss_curves.png
- per-run training_metrics_correlation_matrix.png
- per-run test_predictions_visualized/ if visualization cell was run
- weights/best.pt and weights/last.pt if INCLUDE_WEIGHTS=True
"""

    zf.writestr("README_RESULTS_PACKAGE.txt", readme_text)

# =========================================================
# Report ZIP info
# =========================================================
zip_size_mb = zip_path.stat().st_size / (1024 * 1024)

print("\nDone packaging results.")
print(f"ZIP file: {zip_path}")
print(f"ZIP size: {zip_size_mb:.2f} MB")

print("\nYou can download this file from Kaggle output:")
print(zip_path)

# Optional: show summary table again
print("\nSummary preview:")
display(summary_df)
